# 3.1 - Querying Files

This notebook demonstrates file querying in Databricks, including reading CSV files and exploring file-based datasets.

In [0]:
%sql
-- Check the current catalog and database context
-- This query returns the active catalog and schema (database) for the session
-- Useful for verifying your working context before querying tables
SELECT current_catalog(), current_database();

In [0]:
%sql
-- Switch to the correct catalog and database
-- Ensures all subsequent table operations use the appropriate storage location
USE CATALOG databricks_demo;
USE DATABASE default;

## **CTAS METHOD**

### Reading data directly from the file storege and with default schema.

In [0]:
%sql
-- Create a table from CSV file with all book records
-- Uses CREATE TABLE AS SELECT (CTAS) to load data from file storage
-- The file is referenced via Unity Catalog Volumes, not S3
-- Options ensure the header row is used as column names, schema is inferred, and ',' is the delimiter
CREATE OR REPLACE TABLE tb_books AS
SELECT *
FROM csv.`/Volumes/databricks_demo/default/files_data/books.csv`
WITH (header = 'true', inferSchema = 'true', delimiter = ',');

In [0]:
# Preview the first record from the books table
# Using display() to show result interactively
# LIMIT 1 prevents loading large datasets accidentally

result = spark.sql("SELECT * FROM tb_books limit 1")
display(result)

In [0]:
%sql
-- Read customer data into a table from CSV file
-- CREATE OR REPLACE TABLE loads the entire file into tb_users
-- Options: header names, automatic schema detection, comma-separated values
-- Quick setup for downstream analysis
CREATE OR REPLACE TABLE tb_users AS
SELECT *
FROM csv.`/Volumes/databricks_demo/default/files_data/customers.csv`
WITH (header = 'true', inferSchema = 'true', delimiter = ',');

In [0]:
%sql
-- Get detailed info about tb_users table
-- DESCRIBE EXTENDED outputs column names, data types, metadata, and file location
-- Useful for schema verification before actual analytic queries
DESCRIBE EXTENDED tb_users;

In [0]:
%sql
-- Load Order data from CSV file into tb_orders table
-- Leverages CTAS pattern for rapid file-to-table conversion
-- CSV options: header identifies columns, schema auto-inferred, comma delimiter
CREATE OR REPLACE TABLE tb_orders AS
SELECT *
FROM csv.`/Volumes/databricks_demo/default/files_data/orders.csv`
WITH (header = 'true', inferSchema = 'true', delimiter = ',');

-- Preview order records for validation
SELECT * FROM tb_orders LIMIT 10;

In [0]:
# Read books data from CSV using Spark API
# .option("header", "true") uses the first row as columns
# .option("inferSchema", "true") lets Spark auto-detect column types
# .load() takes the Volumes path for file access
# This creates a Spark DataFrame for programmatic processing

df_books = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/databricks_demo/default/files_data/books.csv")

In [0]:
# Read multiple CSV files (batch processing)
# Wildcard pattern matches all CSVs in target directory
# Schema detection and header mapping are applied for all files
# Good for ingesting partitioned or continued datasets

df_users = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/databricks_demo/default/files_data/csv_files/*.csv")

display(df_users.limit(2))  # Preview first 2 rows

In [0]:
%sql
-- Ingest multiple CSV files and capture file source metadata
-- input_file_name() adds a column with the absolute file path
-- Useful for provenance tracking or batch audits
CREATE OR REPLACE TABLE tb_users
AS
SELECT *, input_file_name() AS source_file
FROM csv.`/Volumes/databricks_demo/default/files_data/csv_files/*.csv`
WITH (header = 'true', inferSchema = 'true', delimiter = ',');

In [0]:
%sql
-- Show 10 sampled user records for quality checks
-- Ensures data load matches expectations; visually spot missing fields or parse issues
SELECT * FROM tb_users limit 10;

### Reading data from storage files and with predefined schema while loading into the table

In [0]:
%sql
-- Create temporary view from CSV with custom options
-- Useful for ad hoc queries without persisting physical tables
-- Options: file path, using header as schema, schema inference
CREATE OR REPLACE TEMPORARY VIEW vw_tm_books_with_using
USING CSV
OPTIONS (
  path = '/Volumes/databricks_demo/default/files_data/books.csv',
  header = 'true',
  inferSchema = 'true'
);

-- Sample view contents (first two rows)
SELECT * FROM vw_tm_books_with_using LIMIT 2;

In [0]:
%sql
-- Display extended metadata for tb_books
-- Reveals detailed schema, data types, file storage, and lineage info
DESCRIBE EXTENDED tb_books;

In [0]:
%sql
-- Inspect extended details for temporary view
-- Includes schema, column types, upstream lineage for view auditing
DESCRIBE EXTENDED vw_tm_books_with_using;

## Query Filing Using read_files method

In [0]:
%sql
-- Create a table using read_files function with file metadata
-- Reads all CSVs matching path, adds file path, name, and size to each record for audit/provenance
CREATE OR REPLACE TABLE tb_users_with_read_files
SELECT *,
      _metadata.file_path AS source_file,
      _metadata.file_name AS source_file_name,
      _metadata.file_size AS source_file_size
FROM read_files(
  '/Volumes/databricks_demo/default/files_data/csv_files/*.csv',
  format => 'csv',
  header => 'true',
  delimiter => ',' 
);

In [0]:
%sql
-- View the entire file-metadata enriched user records
-- Useful for QC, auditing file loads, or downstream transformation
select * from tb_users_with_read_files